# 4 — Full Pipeline Integration

Tests the integrated system with both components (Bingham filter + T³ QAN) on the 2D baseline arena (not running the experiment).

In [ ]:
!ls -d /workspace/CAN_Path_Integration_3D_model/network 2>/dev/null && echo found || find / -maxdepth 4 -type d -name network 2>/dev/null

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

import os
root = os.path.abspath("../..")
print("Notebook CWD :", os.getcwd())
print("Computed root:", root)
print("network here?:", os.path.isdir(os.path.join(root, "network")))


import sys
from pathlib import Path
root = Path.cwd().resolve()
while not (root / "network").is_dir() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("Added:", root)

sys.path.insert(0, os.path.abspath("../.."))

from network.visualize3D import visualize_trajectory_projections, plot_pi_error
from experiments.arena_2d import Arena2DExperiment, Arena2DConfig
from analysis.scoring import score_2d
from metrics import wrapped_angle_diff
from config import RunConfig, NetworkConfig, ExperimentConfig
from experiments.arena_2d import Arena2DExperiment




Notebook CWD : /
Computed root: /
network here?: False
Added: /


ModuleNotFoundError: No module named 'network'

In [ ]:
# config + experiment
g_vec = np.array([0., 0., -9.81])
cfg = RunConfig(
    network=NetworkConfig(build_connectivity=False),                 # validated defaults
    experiment=Arena2DConfig(n_steps=100000), # Arena2DConfig → has run_name
)
exp = Arena2DExperiment(cfg, record=True)    # config, not g


## Set up the arena

In [ ]:
scale = cfg.experiment.scale
print("scale:", scale, " speed:", cfg.experiment.speed,
      " target_speed_rad:", getattr(cfg, "target_speed_rad", None))
print("commanded theta_dot = scale*speed =", scale * cfg.experiment.speed, "rad/step")
print("ceiling = 0.0168  -> saturated?" , scale*cfg.experiment.speed > 0.0168)

scale: 13.089969389957473  speed: 0.0007639437268410976  target_speed_rad: None
commanded theta_dot = scale*speed = 0.01 rad/step
ceiling = 0.0168  -> saturated? False


In [ ]:
# run + save
print("Running ...")
t0 = time.time()
result = exp.run_experiment(g=g_vec)
exp.save(result, f"runs/{cfg.experiment.run_name}.npz")
print(f"Saved runs/{cfg.experiment.run_name}.npz  ({time.time()-t0:.1f}s)")

# everything from the result object (single source of truth)
world_pos  = result.world_pos
torus_gt   = result.torus_gt
theta_hist = result.theta_hist
gap        = result.gap_hist
n_hat_hist = result.n_hat_hist
T          = cfg.experiment.n_steps

print(f"Filter gap  t=100: {gap[99]:.4f}  t=500: {gap[499]:.4f}  t={T}: {gap[-1]:.4f}  target > 2 by t=500")
print(f"n_hat at t={T}: {n_hat_hist[-1]}  (target [0, 0, 1])")
print(f"theta_3 decoded range: [{theta_hist[:, 2].min():.4f}, {theta_hist[:, 2].max():.4f}]  target ~[0, 0]")
print(f"MADE error  full: {result.mean_norm_error:.4f}  after t=200: {result.norm_error[200:].mean():.4f}")

Running ...


NameError: name 'cfg' is not defined

In [ ]:
import numpy as np
from metrics import wrapped_angle_diff

m  = exp.qan.manifold.metric
sc = cfg.experiment.scale

W = 200
gt_d  = wrapped_angle_diff(torus_gt[W:],   torus_gt[:-W])[:, :2]
dec_d = wrapped_angle_diff(theta_hist[W:], theta_hist[:-W])[:, :2]
sg, sd = np.linalg.norm(gt_d,1), np.linalg.norm(dec_d,1)
print("windowed cos:", np.median((dec_d*gt_d).sum(1)/(sd*sg+1e-9)))

# (a) the two scale candidates + what to_phase actually is (linear? a twist matrix?)
print("scale used       :", sc)
print("2pi/grid_spacing :", 2*np.pi/cfg.experiment.grid_spacing, " <- correct")
print("2pi/env_size     :", 2*np.pi/cfg.experiment.env_size,     " <- 'bug' value")
for vv in (np.array([[1.,0.]]), np.array([[0.,1.]]), np.array([[2.,0.]])):
    print("  to_phase", vv.ravel(), "->", np.round(m.to_phase(vv).ravel(), 4))

# (b) did the bump integrate at the right SPEED and DIRECTION? -> tests BOTH pasted bugs
gt_vel  = wrapped_angle_diff(torus_gt[1:],   torus_gt[:-1])[:, :2]   # how it SHOULD move
dec_vel = wrapped_angle_diff(theta_hist[1:], theta_hist[:-1])[:, :2] # how it ACTUALLY moved
sp_gt, sp_dec = np.linalg.norm(gt_vel,1), np.linalg.norm(dec_vel,1)
mask = sp_gt > 1e-4
print("speed ratio decoded/gt :", round(np.median(sp_dec[mask]/sp_gt[mask]),3),
      " (1=correct, ~.15=6.7x bug, ~.88=gain65)")
cos = (dec_vel*gt_vel).sum(1)/(sp_dec*sp_gt+1e-12)
print("direction cos(dec,gt)  :", round(np.median(cos[mask]),3),
      " (1=aligned, <1=mis-rotated => twist)")

# (c) what KIND of tracking error: constant offset vs accumulating drift?
e = np.linalg.norm(wrapped_angle_diff(theta_hist, torus_gt), axis=1)
print("err t<500:", round(e[:500].mean(),3), "| t~20k:", round(e[19750:20250].mean(),3),
      "| t>39.5k:", round(e[-500:].mean(),3), "| mean as %period:", round(e.mean()/(2*np.pi),3))



In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(world_pos[:, 0], world_pos[:, 1], lw=0.5, alpha=0.7, color="steelblue")
ax.scatter(*world_pos[0, :2],  color="green", s=60, zorder=5, label="start")
ax.scatter(*world_pos[-1, :2], color="red",   s=60, zorder=5, label="end")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Physical world trajectory (flat 2D arena)")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, _ = visualize_trajectory_projections(
    torus_gt, theta_hist,
    title="2D baseline arena, ground truth vs decoded on the torus manifold",
)
plt.show()


In [ ]:
fig, _ = plot_pi_error(torus_gt, theta_hist)
plt.show()

In [ ]:
data = np.load(f"runs/{cfg.experiment.run_name}.npz")
S = data["S_tot_buffer"]           # (T, N)


In [ ]:
out = score_2d(f"runs/{cfg.experiment.run_name}.npz", norm="minmax")
f, ac = out["f"], out["ac"]
print(f"{f.shape[-1]} cells | occupancy {out['occupancy']:.1%}")

In [ ]:
print("scale:", cfg.scale, " speed:", cfg.speed,
      " target_speed_rad:", getattr(cfg, "target_speed_rad", None))
print("commanded theta_dot = scale*speed =", cfg.scale * cfg.speed, "rad/step")
print("ceiling = 0.0168  -> saturated?" , cfg.scale*cfg.speed > 0.0168)